# Tier-2 解冻网格:loss × 冻结比例(锁定 swinv2 + mpnet, cosine)

运行:Cell1→6(设置)→ Cell7(网格)。和 LoRA notebook 并行跑。

- 5 损失 × 耦合 f∈{0,1/3,2/3,1}(冻"前 f"比例;f=1 复用冻结筛选不重训)+ 历史候选 L2@vis0.5/txt0.34。
- 每个 run set_seed(42) 同起点;断点续跑;OOM 自动降批;EPOCHS=100/40/30。
- 需要 `clip_encoder_screen_frozen/loss_frozen_swinv2_mpnet.csv`(复用 f=1 那列)。

In [1]:
# Cell 1. Config — Tier-2 解冻网格 (锁定 swinv2 + mpnet, cosine, 充分训练)
import warnings, os, logging, gc, time, re
from contextlib import nullcontext
warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["HF_HOME"]="/root/autodl-tmp/hf_cache"
os.environ["HF_ENDPOINT"]="https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"]="1"; os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
from pathlib import Path
Path("/root/autodl-tmp/hf_cache").mkdir(parents=True, exist_ok=True)
LOCAL_ONLY=False

SPLIT_ROOT=Path("splits_clip/cv5"); CLASSES_CSV=Path("splits_clip/classes.csv")
IMG_ROOT=Path("."); ABL_FOLD=0
OUTDIR=Path("clip_tier2_freeze"); OUTDIR.mkdir(exist_ok=True)

VIS_ID, VNAME="microsoft/swinv2-base-patch4-window8-256","swinv2_base"
TXT_ID, TNAME="sentence-transformers/all-mpnet-base-v2","mpnet"
DEFAULT_TEXT_ID, DEFAULT_TEXT = TXT_ID, TNAME

FIXED_LOSS="L1_ce"; SIM="cosine"
FREEZE_VIS, FREEZE_TXT = 1.0, 1.0               # Cell7 会按需设;1.0=全冻结(冻前段)
PROJ_DIM=512
EPOCHS, MIN_EPOCHS, PATIENCE = 100, 40, 30      # 充分训练
BATCH, OOM_BATCH = 16, 4
LR_HEAD, LR_ENC, MAX_TOK, GRAD_CLIP, SEED = 1e-3, 1e-5, 32, 1.0, 42
AUGMENT=True; USE_SAMPLER=True
print(f"OUTDIR={OUTDIR} | fold {ABL_FOLD} | LOCKED {VNAME}+{TNAME} | epochs={EPOCHS}/{MIN_EPOCHS}/{PATIENCE}")

OUTDIR=clip_tier2_freeze | fold 0 | LOCKED swinv2_base+mpnet | epochs=100/40/30


In [2]:
# Cell 2. Imports + data + dataset/loader (same as Method 1)
import numpy as np, pandas as pd, random, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from torchvision import transforms
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from collections import deque
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed(SEED)
DEV="cuda" if torch.cuda.is_available() else "cpu"; BF16=(DEV=="cuda" and torch.cuda.is_bf16_supported())
print("device:",DEV,"| bf16:",BF16)

CLASSES=list(pd.read_csv(CLASSES_CSV)["caption"]); CIDX={c:i for i,c in enumerate(CLASSES)}; NCLS=len(CLASSES)
def load_fold(k):
    out={}
    for nm in ["train","val","test"]:
        df=pd.read_csv(SPLIT_ROOT/f"fold{k}"/f"{nm}.csv")
        df["label"]=df["caption_norm"].map(CIDX); out[nm]=df.reset_index(drop=True)
    return out
def metrics(true,pred):
    true=list(true); pred=list(pred)
    labs=sorted(set(true))   # 只在测试集中实际出现的类上算 macro(标准做法,不被缺席类拉低)
    acc=accuracy_score(true,pred)
    pma,rma,fma,_=precision_recall_fscore_support(true,pred,labels=labs,average="macro",zero_division=0)
    pw,rw,fw,_=precision_recall_fscore_support(true,pred,labels=labs,average="weighted",zero_division=0)
    return {"acc":acc,"macroF1":fma,"macroP":pma,"macroR":rma,"weightedF1":fw}

AUG=transforms.Compose([transforms.RandomHorizontalFlip(),
                        transforms.ColorJitter(0.2,0.2,0.2,0.02),
                        transforms.RandomRotation(8)])
class ImgDS(Dataset):
    def __init__(self,df,proc,train):
        self.df=df; self.proc=proc; self.train=train and AUGMENT
        self.y=torch.tensor(df["label"].values)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        img=Image.open(IMG_ROOT/self.df.iloc[i]["image"]).convert("RGB")
        if self.train: img=AUG(img)
        pix=self.proc(images=img,return_tensors="pt")["pixel_values"][0]
        return pix, self.y[i], i
def vis_pool(enc,pix):
    m=getattr(enc,"vision_model",enc); h=m(pixel_values=pix).last_hidden_state
    if h.dim()==4: h=h.flatten(2).transpose(1,2)
    return h.mean(1)
def make_loader(df,proc,train):
    ds=ImgDS(df,proc,train)
    if train and USE_SAMPLER:
        cnt=df["label"].value_counts(); w=df["label"].map(lambda l:1.0/cnt[l]).values
        sm=WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double),num_samples=len(df),replacement=True)
        return DataLoader(ds,batch_size=BATCH,sampler=sm,num_workers=0)
    return DataLoader(ds,batch_size=BATCH,shuffle=train,num_workers=0)
print("data ready, NCLS=",NCLS)


device: cuda | bf16: True
data ready, NCLS= 30


In [3]:
# Cell 3. Loss implementations -- the heart of this notebook
class LossFunctions:
    # All losses operate on: logits [B, NCLS] from cosine*scale, labels [B], 
    # image_emb [B, D] normalized, text_emb [NCLS, D] normalized, batch_pos [B, D] = text_emb[labels]
    # class_counts [NCLS] long-tail prior info

    @staticmethod
    def l0_infonce(logits, labels, ie, te, class_counts):
        # symmetric InfoNCE on (image -> text) and (text -> image)
        tpos = te[labels]
        scale = logits.max().item() / max((ie @ te.t())[0,0].item(), 1e-8) if False else None
        # Reconstruct from scratch for clarity
        return F.cross_entropy(logits, labels)  # InfoNCE here = CE over class-prompts when using same logits matrix
        # Note: the real InfoNCE you trained with is symmetric (img->txt + txt->img),
        # but for fold0 ablation we use single-direction CE on the same cosine logits matrix.

    @staticmethod  
    def l0_infonce_symmetric(logits, labels, ie, te, class_counts):
        # True symmetric InfoNCE: image to its positive text + positive text to image
        tpos = te[labels]   # [B, D]
        lc_i2t = logits[:, labels]  # but this is wrong shape; use direct dot
        # Compute pairwise sim between batch_img and batch_pos
        s = ie.size(0)
        lc = (logits[range(s)].t()[labels].t()) if False else None
        # Simplest: standard image-text contrastive with batch-as-negatives
        # logits_i2t[i,j] = sim(img_i, txt_pos_j) where txt_pos_j = text_emb[labels[j]]
        scale = (logits.max() / (ie@te.t()).max()).item() if (ie@te.t()).max()>0 else 100.0
        lc = scale * (ie @ tpos.t())
        tgt = torch.arange(ie.size(0), device=ie.device)
        return 0.5 * (F.cross_entropy(lc, tgt) + F.cross_entropy(lc.t(), tgt))

    @staticmethod
    def l1_ce(logits, labels, ie, te, class_counts):
        return F.cross_entropy(logits, labels)

    @staticmethod
    def l2_balanced_softmax(logits, labels, ie, te, class_counts):
        adj = torch.log(class_counts.float() + 1e-8).to(logits.device).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l3_logit_adjust(logits, labels, ie, te, class_counts, tau=1.0):
        prior = (class_counts.float() / class_counts.sum()).to(logits.device)
        adj = (tau * torch.log(prior + 1e-8)).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l4_focal(logits, labels, ie, te, class_counts, gamma=2.0):
        ce = F.cross_entropy(logits, labels, reduction="none")
        p = (-ce).exp()  # p_y = exp(-CE)
        return ((1-p)**gamma * ce).mean()

    @staticmethod
    def l5_cb_loss(logits, labels, ie, te, class_counts, beta=0.99):
        eff_num = (1.0 - beta**class_counts.float()) / (1.0 - beta + 1e-8)
        weights = (1.0 / (eff_num + 1e-8))
        weights = weights / weights.sum() * NCLS  # normalize so mean weight ~ 1
        return F.cross_entropy(logits, labels, weight=weights.to(logits.device))

    @staticmethod
    def l6_ldam(logits, labels, ie, te, class_counts, max_m=0.5, s=30.0):
        m_list = 1.0 / torch.sqrt(torch.sqrt(class_counts.float() + 1e-8))
        m_list = m_list * (max_m / m_list.max())
        m_list = m_list.to(logits.device)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        margin_batch = m_list[labels].unsqueeze(1)
        logits_m = logits - index.float() * margin_batch
        return F.cross_entropy(logits_m * s / logits.max().clamp(min=1.0), labels)

    @staticmethod
    def l7_supcon(logits, labels, ie, te, class_counts, temp=0.07):
        # Supervised contrastive on image features: same-class samples in batch are positives
        z = F.normalize(ie, dim=-1)
        sim = z @ z.t() / temp  # [B, B]
        # mask diagonal
        mask_diag = torch.eye(z.size(0), device=z.device, dtype=torch.bool)
        sim.masked_fill_(mask_diag, -1e9)
        # positive mask: same label, not self
        labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_mask = labels_eq & ~mask_diag
        # log_prob
        logits_max, _ = sim.max(dim=1, keepdim=True)
        sim_norm = sim - logits_max.detach()
        exp_sim = sim_norm.exp()
        log_prob = sim_norm - torch.log(exp_sim.sum(1, keepdim=True) + 1e-12)
        # mean log_prob over positives (skip samples with no positives)
        n_pos = pos_mask.sum(1)
        valid = n_pos > 0
        if valid.sum() == 0: 
            return F.cross_entropy(logits, labels)  # fallback
        mean_log_prob = (pos_mask.float() * log_prob).sum(1)[valid] / n_pos[valid].float()
        loss_supcon = -mean_log_prob.mean()
        # Add the cosine-CE for classifier supervision (otherwise model has no class anchor)
        return 0.5 * loss_supcon + 0.5 * F.cross_entropy(logits, labels)

    @staticmethod
    def l8_arcface(logits, labels, ie, te, class_counts, m=0.5, s=30.0):
        # logits are scale * cosine; recover cosine
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        theta = cos_theta.acos()
        # Only apply margin to ground-truth class
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        theta_m = theta + index.float() * m
        cos_theta_m = theta_m.cos()
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

    @staticmethod
    def l9_cosface(logits, labels, ie, te, class_counts, m=0.4, s=30.0):
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        cos_theta_m = cos_theta - index.float() * m
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

LOSS_FN = {
    "L0_infonce":            LossFunctions.l0_infonce_symmetric,
    "L1_ce":                 LossFunctions.l1_ce,
    "L2_balanced_softmax":   LossFunctions.l2_balanced_softmax,
    "L3_logit_adjust_t10":   LossFunctions.l3_logit_adjust,
    "L4_focal_g2":           LossFunctions.l4_focal,
    "L5_cb_loss_b099":       LossFunctions.l5_cb_loss,
    "L6_ldam":               LossFunctions.l6_ldam,
    "L7_supcon":             LossFunctions.l7_supcon,
    "L8_arcface_m05":        LossFunctions.l8_arcface,
    "L9_cosface_m04":        LossFunctions.l9_cosface,
}
print("loss functions:", list(LOSS_FN.keys()))


loss functions: ['L0_infonce', 'L1_ce', 'L2_balanced_softmax', 'L3_logit_adjust_t10', 'L4_focal_g2', 'L5_cb_loss_b099', 'L6_ldam', 'L7_supcon', 'L8_arcface_m05', 'L9_cosface_m04']


In [4]:
# Cell 4. Model (same architecture, no logit_adjust param -- losses see raw cosine logits)
def partial_unfreeze(model, freeze_frac):
    for p in model.parameters(): p.requires_grad=False
    if freeze_frac>=0.999:
        return 0   # FROZEN: encoder fully frozen, only projection heads train (linear probe)
    names=[n for n,_ in model.named_parameters()]
    li=[int(m.group(1)) for n in names for m in [re.search(r"encoder\.layer\.(\d+)\.",n)] if m]
    si=[int(m.group(1)) for n in names for m in [re.search(r"encoder\.layers\.(\d+)\.",n)] if m]
    ntr=0
    if li:
        cut=int(round(max(li)*freeze_frac))
        for n,p in model.named_parameters():
            m=re.search(r"encoder\.layer\.(\d+)\.",n)
            if (m and int(m.group(1))>=cut) or any(k in n.lower() for k in ["layernorm","pooler"]): p.requires_grad=True; ntr+=p.numel()
    elif si:
        cut=int(round((max(si)+1)*freeze_frac))
        for n,p in model.named_parameters():
            m=re.search(r"encoder\.layers\.(\d+)\.",n)
            if (m and int(m.group(1))>=cut) or "layernorm" in n.lower(): p.requires_grad=True; ntr+=p.numel()
    else:
        for p in model.parameters(): p.requires_grad=True; ntr+=p.numel()
    return ntr

class CLIPRetriever(nn.Module):
    def __init__(self, vis_id, txt_id, proc, D=PROJ_DIM):
        super().__init__()
        self.venc=AutoModel.from_pretrained(vis_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        self.tenc=AutoModel.from_pretrained(txt_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        with torch.no_grad():
            dummy=proc(images=Image.new("RGB",(256,256)),return_tensors="pt")["pixel_values"]
            dv=vis_pool(self.venc,dummy).shape[-1]
        dt=self.tenc.config.hidden_size
        self.ip=nn.Sequential(nn.Linear(dv,D),nn.GELU(),nn.Linear(D,D))
        self.tp=nn.Sequential(nn.Linear(dt,D),nn.GELU(),nn.Linear(D,D))
        self.scale=nn.Parameter(torch.tensor(float(np.log(1/0.07))))
        nv=partial_unfreeze(self.venc,FREEZE_VIS); nt=partial_unfreeze(self.tenc,FREEZE_TXT)
        print(f"   unfrozen: vision {nv:,} | text {nt:,}")
    def enc_text(self, ids, mask):
        o=self.tenc(input_ids=ids,attention_mask=mask).last_hidden_state
        m=mask.unsqueeze(-1).float(); return self.tp((o*m).sum(1)/m.sum(1).clamp(min=1))
    def enc_img(self, pix): return self.ip(vis_pool(self.venc,pix))
    def forward_features(self, pix, tids, tmask):
        ie=self.enc_img(pix); te=self.enc_text(tids,tmask)
        s=self.scale.exp().clamp(max=100.0)
        ien=F.normalize(ie,dim=-1); ten=F.normalize(te,dim=-1)
        logits = s * ien @ ten.t()
        return logits, ien, ten
    @torch.no_grad()
    def predict(self, pix, tids, tmask):
        lg, _, _ = self.forward_features(pix, tids, tmask)
        return lg
print("model class ready")


model class ready


In [5]:
# Cell 5. Single-loss training function
def train_run(loss_name, vis_id, txt_id, proc, tok, txt_ids, txt_mask, fold, batch):
    set_seed(SEED)   # 每个 run 从同一随机起点:在建模型/读数据之前重置 -> init+数据顺序对所有配置一致
    F_=load_fold(fold)
    model=CLIPRetriever(vis_id, txt_id, proc).to(DEV)
    head=[p for n,p in model.named_parameters() if p.requires_grad and ("ip." in n or "tp." in n or n=="scale")]
    enc=[p for n,p in model.named_parameters() if p.requires_grad and not("ip." in n or "tp." in n or n=="scale")]
    grps=[{"params":head,"lr":LR_HEAD}]+([{"params":enc,"lr":LR_ENC}] if enc else [])
    opt=torch.optim.AdamW(grps,weight_decay=0.01)

    cnt=F_["train"]["label"].value_counts()
    class_counts=torch.tensor([cnt.get(i,1) for i in range(NCLS)], dtype=torch.long, device=DEV)
    print(f"   class_counts: min={class_counts.min().item()} max={class_counts.max().item()}")

    loss_fn = LOSS_FN[loss_name]
    dl=make_loader(F_["train"],proc,True)
    TI,TM=txt_ids.to(DEV),txt_mask.to(DEV)
    def ev(split):
        model.eval(); P=[]; T=[]
        for pix,y,_ in DataLoader(ImgDS(F_[split],proc,False),batch_size=batch):
            lg=model.predict(pix.to(DEV),TI,TM); P+=lg.argmax(1).cpu().tolist(); T+=y.tolist()
        return T,P

    win=deque(maxlen=3); best=-1; bstate=None; bad=0; bep=-1
    nan_streak=0
    for ep in range(EPOCHS):
        model.train(); run=0; st=0
        for pix,y,_ in dl:
            opt.zero_grad()
            ctx=torch.autocast("cuda",dtype=torch.bfloat16) if BF16 else nullcontext()
            with ctx:
                logits, ie, te = model.forward_features(pix.to(DEV),TI,TM)
                loss = loss_fn(logits, y.to(DEV), ie, te, class_counts)
            if not torch.isfinite(loss): 
                nan_streak += 1
                if nan_streak > 50: 
                    print(f"   ABORT: loss non-finite for 50 batches")
                    return None
                continue
            nan_streak = 0
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
            opt.step()
            with torch.no_grad(): model.scale.clamp_(max=float(np.log(100.0)))
            run+=loss.item(); st+=1
        T,P=ev("val"); m=metrics(T,P); sc=m["acc"]+m["macroF1"]; win.append(sc); sm=sum(win)/len(win)
        if sm>best: best=sm; bep=ep; bad=0; bstate={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        elif ep>=MIN_EPOCHS: bad+=1
        if ep%5==0 or ep==EPOCHS-1: print(f"   ep{ep:03d} loss={run/max(1,st):.3f} val_acc={m['acc']:.3f} val_mF1={m['macroF1']:.3f}")
        if ep>=MIN_EPOCHS and bad>=PATIENCE: print(f"   early stop @ep{ep} (best @ep{bep})"); break

    if bstate: model.load_state_dict(bstate)
    T,P=ev("test"); fm=metrics(T,P)
    del model,opt,dl; gc.collect(); torch.cuda.empty_cache() if DEV=="cuda" else None
    return fm


In [6]:
# Cell 6. Helpers: text tensors + run_combo (OOM-safe)
from transformers import AutoTokenizer, AutoImageProcessor
print(f">>> GPU: {torch.cuda.get_device_name(0)} | bf16={BF16}") if DEV=="cuda" else print(">>> NO GPU")
def text_tensors(txt_id):
    tok=AutoTokenizer.from_pretrained(txt_id, local_files_only=LOCAL_ONLY)
    e=tok(CLASSES,padding="max_length",truncation=True,max_length=MAX_TOK,return_tensors="pt")
    return tok, e["input_ids"], e["attention_mask"]
def run_combo(vis_id, txt_id):
    proc=AutoImageProcessor.from_pretrained(vis_id, local_files_only=LOCAL_ONLY)
    tok,TI,TM=text_tensors(txt_id)
    try:
        return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, ABL_FOLD, BATCH)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache(); gc.collect(); print(f"   OOM -> retry batch {OOM_BATCH}")
            return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, ABL_FOLD, OOM_BATCH)
        raise
print("helpers ready")

>>> GPU: NVIDIA GeForce RTX 4090 | bf16=True
helpers ready


In [7]:
# Cell 7. 解冻网格: 5 损失 × 耦合冻结比例 f(视觉/文本同冻"前 f");cosine; set_seed 同起点; 按 acc
#   f=0 全解冻, f=1 全冻结(复用冻结筛选不重训);另加历史候选 L2 非对称 vis0.5/txt0.34
import time, pandas as pd
pd.set_option("display.float_format", lambda x:f"{x:.4f}")
EPOCHS, MIN_EPOCHS, PATIENCE = 100, 40, 30
SIM="cosine"
LOSSES=["L9_cosface_m04","L1_ce","L4_focal_g2","L2_balanced_softmax","L7_supcon"]
T=1/3
FVALS=[0.0, T, 2*T, 1.0]
GG=OUTDIR/"tier2_freeze_grid.csv"
COLS=["loss","freeze_vis","freeze_txt","vision","text","sim","status","minutes","acc","macroF1","macroP","macroR","weightedF1"]
done=set()
if GG.exists():
    _d=pd.read_csv(GG)
    done={(r["loss"],round(float(r["freeze_vis"]),2),round(float(r["freeze_txt"]),2)) for _,r in _d[_d["status"]=="OK"].iterrows()}

def write_row(rec): pd.DataFrame([rec])[COLS].to_csv(GG,mode="a",header=not GG.exists(),index=False)

def run_one(ln, fv, ft):
    global FIXED_LOSS, FREEZE_VIS, FREEZE_TXT
    key=(ln,round(fv,2),round(ft,2))
    if key in done: print(f"[SKIP] {ln} vis={fv:.2f}/txt={ft:.2f}"); return
    FIXED_LOSS=ln; FREEZE_VIS, FREEZE_TXT = fv, ft
    print("="*64); print(f"{ln} | freeze vis={fv:.3f}/txt={ft:.3f} (冻前段)")
    t0=time.time(); base={"loss":ln,"freeze_vis":round(fv,4),"freeze_txt":round(ft,4),"vision":"swinv2_base","text":"mpnet","sim":"cosine"}
    try:
        fm=run_combo(VIS_ID,TXT_ID)
        rec={**base,"status":"OK","minutes":round((time.time()-t0)/60,1),**{k:round(v,4) for k,v in fm.items()}}
        print(f"   DONE acc={fm['acc']:.4f} macroF1={fm['macroF1']:.4f}")
    except Exception as e:
        print(f"   FAILED: {type(e).__name__}: {str(e)[:220]}")
        rec={**base,"status":f"FAILED_{type(e).__name__}","minutes":round((time.time()-t0)/60,1),
             "acc":float('nan'),"macroF1":float('nan'),"macroP":float('nan'),"macroR":float('nan'),"weightedF1":float('nan')}
    write_row(rec); done.add(key)

# f=1.0(全冻结)复用冻结 loss 筛选,不重训
FROZEN_CSV=Path("clip_encoder_screen_frozen/loss_frozen_swinv2_mpnet.csv")
if FROZEN_CSV.exists():
    fz=pd.read_csv(FROZEN_CSV); fz=fz[fz["status"]=="OK"]
    for ln in LOSSES:
        if (ln,1.0,1.0) in done: continue
        row=fz[fz["loss"]==ln]
        if len(row):
            r0=row.iloc[0]
            rec={"loss":ln,"freeze_vis":1.0,"freeze_txt":1.0,"vision":"swinv2_base","text":"mpnet","sim":"cosine",
                 "status":"OK","minutes":0.0,"acc":r0["acc"],"macroF1":r0["macroF1"],
                 "macroP":r0["macroP"],"macroR":r0["macroR"],"weightedF1":r0["weightedF1"]}
            write_row(rec); done.add((ln,1.0,1.0)); print(f">>> f=1.0 复用: {ln} acc={r0['acc']:.4f}")
        else: print(f"!! {ln} 不在 frozen CSV")
else:
    print("!! 没找到 clip_encoder_screen_frozen/loss_frozen_swinv2_mpnet.csv —— f=1.0 会重跑")

# 主网格:耦合 f ∈ {0, 1/3, 2/3}
for ln in LOSSES:
    for f in FVALS:
        if f>=0.999: continue
        run_one(ln, f, f)

# 历史候选:L2 非对称 vis0.5 / txt0.34
run_one("L2_balanced_softmax", 0.5, 0.34)

# 汇总
g=pd.read_csv(GG); g=g[g["status"]=="OK"].copy()
cp=g[(g["freeze_vis"]-g["freeze_txt"]).abs()<1e-6]
print("\n"+"="*74); print("解冻网格 — 耦合 loss × f (swinv2+mpnet, cosine, fold0) — acc"); print("="*74)
print(cp.pivot_table(index="loss",columns=cp["freeze_vis"].round(2),values="acc",aggfunc="max").to_string())
print("\n--- macroF1 ---")
print(cp.pivot_table(index="loss",columns=cp["freeze_vis"].round(2),values="macroF1",aggfunc="max").to_string())
asym=g[(g["freeze_vis"]-g["freeze_txt"]).abs()>=1e-6]
if len(asym):
    print("\n--- 历史非对称候选 ---")
    print(asym[["loss","freeze_vis","freeze_txt","acc","macroF1","weightedF1"]].to_string(index=False))
b=g.sort_values("acc",ascending=False).iloc[0]
print(f"\n>>> fold0 最优: loss={b['loss']} vis={b['freeze_vis']:.2f}/txt={b['freeze_txt']:.2f} acc={b['acc']:.4f} macroF1={b['macroF1']:.4f}")

>>> f=1.0 复用: L9_cosface_m04 acc=0.8943
>>> f=1.0 复用: L1_ce acc=0.8862
>>> f=1.0 复用: L4_focal_g2 acc=0.8780
>>> f=1.0 复用: L2_balanced_softmax acc=0.8455
>>> f=1.0 复用: L7_supcon acc=0.8618
L9_cosface_m04 | freeze vis=0.000/txt=0.000 (冻前段)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

   unfrozen: vision 25,268,288 | text 36,052,992
   class_counts: min=3 max=66
   ep000 loss=1.776 val_acc=0.468 val_mF1=0.360
   ep005 loss=0.356 val_acc=0.726 val_mF1=0.542
   ep010 loss=0.133 val_acc=0.823 val_mF1=0.757
   ep015 loss=0.163 val_acc=0.823 val_mF1=0.682
   ep020 loss=0.075 val_acc=0.839 val_mF1=0.690
   ep025 loss=0.098 val_acc=0.823 val_mF1=0.719
   ep030 loss=0.086 val_acc=0.839 val_mF1=0.715
   ep035 loss=0.091 val_acc=0.806 val_mF1=0.652
   ep040 loss=0.098 val_acc=0.871 val_mF1=0.738
   ep045 loss=0.090 val_acc=0.871 val_mF1=0.735
   ep050 loss=0.150 val_acc=0.855 val_mF1=0.739
   ep055 loss=0.074 val_acc=0.790 val_mF1=0.690
   ep060 loss=0.071 val_acc=0.871 val_mF1=0.722
   ep065 loss=0.058 val_acc=0.823 val_mF1=0.707
   ep070 loss=0.074 val_acc=0.855 val_mF1=0.727
   ep075 loss=0.082 val_acc=0.855 val_mF1=0.719
   ep080 loss=0.086 val_acc=0.887 val_mF1=0.755
   early stop @ep80 (best @ep50)
   DONE acc=0.8943 macroF1=0.8003
L2_balanced_softmax | freeze vis=0.500

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   unfrozen: vision 84,239,712 | text 57,307,392
   class_counts: min=3 max=66
   ep000 loss=1.952 val_acc=0.532 val_mF1=0.457
   ep005 loss=0.114 val_acc=0.742 val_mF1=0.591
   ep010 loss=0.017 val_acc=0.806 val_mF1=0.637
   ep015 loss=0.033 val_acc=0.839 val_mF1=0.699
   ep020 loss=0.013 val_acc=0.855 val_mF1=0.746
   ep025 loss=0.030 val_acc=0.855 val_mF1=0.749
   ep030 loss=0.028 val_acc=0.855 val_mF1=0.756
   ep035 loss=0.031 val_acc=0.839 val_mF1=0.714
   ep040 loss=0.001 val_acc=0.823 val_mF1=0.666
   ep045 loss=0.001 val_acc=0.806 val_mF1=0.638
   ep050 loss=0.039 val_acc=0.790 val_mF1=0.583
   ep055 loss=0.011 val_acc=0.839 val_mF1=0.671
   ep060 loss=0.002 val_acc=0.855 val_mF1=0.735
   ep065 loss=0.008 val_acc=0.839 val_mF1=0.659
   early stop @ep69 (best @ep22)
   DONE acc=0.8699 macroF1=0.7825

解冻网格 — 耦合 loss × f (swinv2+mpnet, cosine, fold0) — acc
freeze_vis           0.0000  0.3300  0.6700  1.0000
loss                                               
L1_ce                0